# # RAG-Based Customer Support Assistant

  Install libraries

In [1]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers

Import PDF loader

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Bank FAQ.pdf")
documents = loader.load()

print(len(documents))

34


Chunking

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

160


Embeddings + ChromaDB

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings()

db = Chroma.from_documents(chunks, embedding)

print("DB ready")

C:\Users\USER VICTUS\AppData\Local\Temp\ipykernel_6476\3123686993.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings()
C:\Users\USER VICTUS\AppData\Local\Temp\ipykernel_6476\3123686993.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DB ready


Retriever

In [5]:
retriever = db.as_retriever(search_kwargs={"k": 3})

query = "password reset"

docs = retriever.invoke(query)

for doc in docs:
    print("----")
    print(doc.page_content)

----
of the CGST Act, 2017 
for reversal of input
----
Page 
3
 
of 
34
 
 
returns filed by it. Later, in case the 
customer reverts with the GSTIN, 
how should this amendment be 
reflected?
 
8.
 
 
How should the turnover during 
the 
period from July 2017 to March 2018 
be determined for the purposes of 
distribution of ISD credit?
 
As per the Explanation to Section 20 of the CGST Act, 2017, the relevant period 
on the basis of which the ratio of aggregate turnover for distribution of 
ISD
----
for the purpose of repairs, etc. does not constitute a supply. The equipment 
may be moved by the Banks to the 
location of the third party service providers 
and after repairs, the equipment may be moved to a central / regional location 
for the purpose of programming, encryption, reconfiguration, etc. and 
thereafter to that place of business from where the equipment
 
had been sent
 
earlier
. The equipment can be moved between such locations on the basis of a 
‘delivery challan’.
 
16.


LLM Setup

In [6]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# simple local model
pipe = pipeline("text-generation", model="gpt2")

llm = HuggingFacePipeline(pipeline=pipe)

def rag_pipeline(query):
    docs = retriever.invoke(query)
    context = " ".join([doc.page_content for doc in docs])

    prompt = f"Answer based on context:\n{context}\n\nQuestion: {query}"

    return llm.invoke(prompt)

print(rag_pipeline("What is CGST Act?"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

C:\Users\USER VICTUS\AppData\Local\Temp\ipykernel_6476\2746500684.py:7: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer based on context:
of the CGST Act, 2017 
for reversal of input Page 
20
 
of 
34
 
 
and other financial institutions?
 
services.
 
For such services invariably a variety of instruments are used in the 
financial markets. Transactions in such instruments have to be examined on 
the touchstone of definition of ‘supply’ given in Section 7(1) of the CGST Act, 
2017 to see whether such trans
actions would be chargeable to GST. Broadly, the 
following legal provisions would have a bearing on determining the taxability 
of such transactions. claim. Schedule III of the CGST Act
, 2017
 
lists activities or transactions which 
shall be treated neither as a supply of goods nor a supply of services and 
actionable claims ot
her than lottery, betting and gambling are included in the 
said Schedule. 
Thus, only actionable claims in respect of lottery, betting and

Question: What is CGST Act?

[5] 

(1) The CGST Act provides for the imposition of a tax on the supply of

services and service

In [7]:
!pip install langgraph

LangGraph imports

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

GraphState

process_query()

In [9]:
class GraphState(TypedDict):
    query: str
    response: str

In [10]:
def process_query(state):
    query = state["query"]

    response = rag_pipeline(query)

    return {
        "query": query,
        "response": response
    }

In [11]:
workflow = StateGraph(GraphState)

workflow.add_node("process", process_query)

workflow.set_entry_point("process")

workflow.add_edge("process", END)

app = workflow.compile()

In [12]:
result = app.invoke({
    "query": "What is CGST Act?"
})

print(result["response"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer based on context:
of the CGST Act, 2017 
for reversal of input Page 
20
 
of 
34
 
 
and other financial institutions?
 
services.
 
For such services invariably a variety of instruments are used in the 
financial markets. Transactions in such instruments have to be examined on 
the touchstone of definition of ‘supply’ given in Section 7(1) of the CGST Act, 
2017 to see whether such trans
actions would be chargeable to GST. Broadly, the 
following legal provisions would have a bearing on determining the taxability 
of such transactions. claim. Schedule III of the CGST Act
, 2017
 
lists activities or transactions which 
shall be treated neither as a supply of goods nor a supply of services and 
actionable claims ot
her than lottery, betting and gambling are included in the 
said Schedule. 
Thus, only actionable claims in respect of lottery, betting and

Question: What is CGST Act? (p. 22) 

(1) The CGST Act provides that

"any individual is entitled to recover for any reason the

HITL functions

In [13]:
def check_escalation(state):
    response = state["response"]

    if len(response) < 100:
        return "escalate"

    return "end"

In [14]:
def human_escalation(state):
    return {
        "query": state["query"],
        "response": "Query escalated to human support agent."
    }

Final workflow

In [15]:
workflow = StateGraph(GraphState)

# nodes
workflow.add_node("process", process_query)
workflow.add_node("human", human_escalation)

# entry
workflow.set_entry_point("process")

# conditional routing
workflow.add_conditional_edges(
    "process",
    check_escalation,
    {
        "escalate": "human",
        "end": END
    }
)

# human node ends
workflow.add_edge("human", END)

# compile
app = workflow.compile()

Final invoke

In [16]:
result = app.invoke({
    "query": "Explain CGST Act"
})

print(result["response"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer based on context:
of the CGST Act, 2017 
for reversal of input in India. So the liability to discharge GST on such supplies will be required to 
be determi
ned accordingly.
 
 
30.
 
 
Will the second proviso to Rule 28 
apply in the case of a banking 
company that selects the 50% option 
to avail input tax credit set out in 
section 17(4) of the CGST Act, 2017?
 
The second proviso to Rule 28 of the CGST Rules, 2017 states
 
that where the 
recipient is eligible for full input tax credit, the value as declared in the Invoice Page 
20
 
of 
34
 
 
and other financial institutions?
 
services.
 
For such services invariably a variety of instruments are used in the 
financial markets. Transactions in such instruments have to be examined on 
the touchstone of definition of ‘supply’ given in Section 7(1) of the CGST Act, 
2017 to see whether such trans
actions would be chargeable to GST. Broadly, the 
following legal provisions would have a bearing on determining the taxability 
of 